# PsychLing Benchmark: Supplementary Metric Pipeline

This notebook accompanies the ACL submission *PsychLing Benchmark: Psycholinguistic metrics for Human-Model Corpus Comparison in Children and Adults*. It provides a compact, reproducible version of the metric pipeline for two German input texts supplied in the `TEXTS` dictionary below.

The PsychLing Benchmark covers four linguistic levels: textual, lexical, syntactic, and semantic. This supplementary notebook computes the benchmark measures that can be evaluated for a pair of texts: token/type counts, word length, word frequency, lexical diversity (MTLD, HD-D), POS distributions, syntactic complexity, and lexical- and text-level semantic similarity.

In [ ]:
# Input texts. Replace only the string values; labels may be renamed if desired.
TEXTS = {
    "Text 1": """
    ver kleine Hund Dodo 
Lea suchte ihren Hund Dodo , dafür 
klebte sie auf die Wänder Plakate 
mit Dodos Fotos . Auf den Plakatenstand 
Leas Telefonnumer , Leas adrese und alles 
andere . Als Lea nach Hause ging , 
wartete sie traurig auf Dodos anrufe . 
Da rufte jemand an . Es war Lars . 
Er erzählte zu Lea : " Ich habe 
deinen Hund und komme jetzt 
zu dir . Jetzt war Lea fröhlich und 
sprach zu Lars : " ja , ja , komm schnell ! " . 
Er kamm schon und klingelte an 
die Tür . Ein Luschter hatte er im 
Hand . Lea guckte aus dem Fenster . 
Als sie die Tür auf machte , gab 
Lars ihr einen Lutscher . Lea umarmte 
Lars ganz doll und alle waren 
fröhlich und zufrieden . 
    """,
    "Text 2": """
    In der Geschichte verliert ein Kind seinen Hund. 
    Zuerst hängt es einen Zettel an die Wand, um ihn zu suchen. 
    Dann telefoniert das Kind und sieht sehr traurig aus. 
    Während des Gesprächs bekommt es einen Hinweis von jemandem, dass der Hund gefunden wurde. 
    Das Kind ist erleichtert und freut sich. Schließlich kommt eine andere Person mit dem Hund zurück. 
    Beide Kinder sind glücklich und umarmen sich. Der Hund frisst glücklich aus seinem Napf. 
    Die Geschichte zeigt, wie wichtig es ist, Freunde zu haben, die in schwierigen Zeiten helfen, 
    und dass man die Hoffnung nicht aufgeben sollte, wenn etwas verloren geht.
    """,
}


## Runtime setup

Run this cell first in Google Colab or a fresh Python environment. It installs the required Python packages and the German spaCy model used by the lexical, POS, syntactic, and semantic sections.

In [ ]:
# Keep this cell runnable in Colab, but avoid reinstalling packages that are already present.
import importlib.util
import subprocess
import sys

PACKAGE_IMPORTS = {
    "pandas": "pandas",
    "spacy": "spacy",
    "lexical_diversity": "lexical-diversity",
    "numpy": "numpy",
    "scipy": "scipy",
    "torch": "torch",
    "transformers": "transformers",
    "sentence_transformers": "sentence-transformers",
    "sentencepiece": "sentencepiece",
    "stanza": "stanza",
    "nltk": "nltk",
}

missing_packages = [pkg for mod, pkg in PACKAGE_IMPORTS.items() if importlib.util.find_spec(mod) is None]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

import spacy

try:
    spacy.load("de_core_news_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "de_core_news_sm", "-q"])

print("Runtime setup complete.")


In [ ]:
import math
import statistics
from collections import Counter

import pandas as pd
import spacy

try:
    from lexical_diversity import lex_div as ld
except ImportError:
    ld = None

POS_TAGS = ["ADJ", "ADV", "DET", "VERB", "NOUN", "PRON"]
EXCLUDE_POS_LEMMAS = {"hund", "mädchen", "junge", "dodo", "lea", "lars"}


def load_spacy_model(model_name="de_core_news_sm"):
    try:
        return spacy.load(model_name)
    except OSError as exc:
        raise OSError(
            f"spaCy model '{model_name}' is not installed. Install it with: "
            f"python -m spacy download {model_name}"
        ) from exc


nlp = load_spacy_model()


In [ ]:
def tokenize_doc(doc):
    # German capitalization carries information, so nouns/proper names stay as written.
    tokens = []
    for token in doc:
        if not token.is_alpha:
            continue
        if token.pos_ in ("NOUN", "PROPN"):
            tokens.append(token.text)
        else:
            tokens.append(token.text.lower())
    return tokens


def hdd_fallback(tokens, sample_size=42):
    token_total = len(tokens)
    if token_total == 0:
        return 0.0

    sample_size = min(sample_size, token_total)
    freqs = Counter(tokens)

    def p_no_occurrence(total, freq, sample):
        prob = 1.0
        for k in range(sample):
            prob *= (total - freq - k) / (total - k)
        return max(0.0, min(1.0, prob))

    expected_types = sum(
        1.0 - p_no_occurrence(token_total, freq, sample_size)
        for freq in freqs.values()
    )
    return expected_types / sample_size


def mtld_fallback(tokens, ttr_threshold=0.72):
    def ttr(segment):
        return len(set(segment)) / len(segment) if segment else 0.0

    def mtld_calc(sequence):
        factors, start = 0, 0
        while start < len(sequence):
            for i in range(start + 10, len(sequence) + 1):
                if ttr(sequence[start:i]) < ttr_threshold:
                    factors += 1
                    start = i
                    break
            else:
                remaining = sequence[start:]
                if remaining:
                    factors += (1 - ttr(remaining)) / (1 - ttr_threshold)
                break
        return len(sequence) / factors if factors > 0 else 0.0

    return (mtld_calc(tokens) + mtld_calc(tokens[::-1])) / 2 if tokens else 0.0


def lexical_diversity(tokens):
    if not tokens:
        return 0.0, 0.0

    mtld_val, hdd_val = 0.0, 0.0

    if ld is not None:
        try:
            mtld_val = ld.mtld(tokens) or 0.0
        except Exception:
            # Short or unusual transcriptions occasionally trip the package implementation.
            mtld_val = 0.0

        try:
            hdd_val = ld.hdd(tokens) if len(tokens) >= 42 else hdd_fallback(tokens)
        except Exception:
            hdd_val = hdd_fallback(tokens)

    if mtld_val == 0.0:
        mtld_val = mtld_fallback(tokens)
    if hdd_val == 0.0:
        hdd_val = hdd_fallback(tokens)

    return mtld_val, hdd_val


def safe_median(values):
    values = [value for value in values if value is not None]
    return statistics.median(values) if values else 0.0


def safe_mean(values):
    values = [value for value in values if value is not None]
    return statistics.mean(values) if values else 0.0


In [ ]:
def normalize_input_texts(texts):
    if isinstance(texts, dict):
        return [(str(label), str(text)) for label, text in texts.items()]
    return [(f"Text {i}", str(text)) for i, text in enumerate(texts, start=1)]


def count_requested_pos(doc):
    counts = Counter({tag: 0 for tag in POS_TAGS})
    denominator = 0

    for token in doc:
        if not token.is_alpha:
            continue
        lemma = token.lemma_.lower() if token.lemma_ else ""
        if token.pos_ in {"NOUN", "PROPN"} and lemma in EXCLUDE_POS_LEMMAS:
            continue
        if token.pos_ in POS_TAGS:
            counts[token.pos_] += 1
            denominator += 1

    return counts, denominator


def analyze_texts(texts):
    items = normalize_input_texts(texts)
    docs = list(nlp.pipe([text for _, text in items], batch_size=20))

    per_text_rows = []
    all_tokens = []
    total_pos_counts = Counter({tag: 0 for tag in POS_TAGS})
    total_pos_denominator = 0

    for (label, _), doc in zip(items, docs):
        tokens = tokenize_doc(doc)
        mtld_val, hdd_val = lexical_diversity(tokens)
        pos_counts, pos_denominator = count_requested_pos(doc)

        all_tokens.extend(tokens)
        total_pos_counts.update(pos_counts)
        total_pos_denominator += pos_denominator

        row = {
            "text_id": label,
            "tokens_per_text": len(tokens),
            "types_per_text": len(set(tokens)),
            "mtld": mtld_val,
            "hdd": hdd_val,
            "median_word_length": safe_median([len(token) for token in tokens]),
        }
        for tag in POS_TAGS:
            row[f"{tag}_count"] = pos_counts[tag]
            row[f"{tag}_pct"] = (
                pos_counts[tag] / pos_denominator * 100
                if pos_denominator else 0.0
            )
        per_text_rows.append(row)

    token_counts = Counter(all_tokens)
    total_tokens = sum(token_counts.values())
    freq_per_type = [
        count / total_tokens * 1_000_000
        for count in token_counts.values()
    ] if total_tokens else []
    log_freq_per_type = [math.log10(freq) for freq in freq_per_type if freq > 0]

    summary_row = {
        "Total texts": len(items),
        "Total tokens": total_tokens,
        "Mean tokens/text": safe_mean([row["tokens_per_text"] for row in per_text_rows]),
        "Total types": len(token_counts),
        "Mean types/text": safe_mean([row["types_per_text"] for row in per_text_rows]),
        "Median MTLD score": safe_median([row["mtld"] for row in per_text_rows]),
        "Median HD-D score": safe_median([row["hdd"] for row in per_text_rows]),
        "Median word length": safe_median([len(token) for token in all_tokens]),
        "Median log word frequency": safe_median(log_freq_per_type),
    }
    summary = pd.DataFrame([summary_row])

    pos_summary = pd.DataFrame([
        {
            "POS": tag,
            "count": total_pos_counts[tag],
            "percent_of_requested_POS": (
                total_pos_counts[tag] / total_pos_denominator * 100
                if total_pos_denominator else 0.0
            ),
        }
        for tag in POS_TAGS
    ])

    per_text = pd.DataFrame(per_text_rows)
    numeric_cols = summary.select_dtypes(include="number").columns
    summary[numeric_cols] = summary[numeric_cols].round(3)
    per_text = per_text.round(3)
    pos_summary = pos_summary.round(3)

    return summary, pos_summary, per_text


In [ ]:
summary, pos_summary, per_text = analyze_texts(TEXTS)

display(summary)
display(pos_summary)
display(per_text)


## Syntax metrics

This section implements the syntactic measures described in the paper and computed from the CoreNLP parses. It reports sentence length, phrase-structure measures (tree depth, closing nodes, open nodes), and dependency length.

Syntactic parsing uses Stanford CoreNLP with German properties via `stanza`.

In [ ]:
# Configure Stanford CoreNLP for the syntactic measures.
# In a fresh environment this cell downloads CoreNLP and the German models.
import os
import importlib.util
import re
import socket
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("stanza") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "stanza"])
if importlib.util.find_spec("nltk") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nltk"])

import stanza
from nltk.tree import Tree
from stanza.server import CoreNLPClient

# Use an existing CoreNLP installation when CORENLP_HOME is set.
# Otherwise, install/use a local folder next to this notebook.
def find_open_port(host="localhost"):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return sock.getsockname()[1]


def display_path(path):
    try:
        return str(Path(path).resolve().relative_to(Path.cwd().resolve()))
    except ValueError:
        return Path(path).name


CORENLP_HOME = os.environ.get("CORENLP_HOME") or str(Path.cwd() / "corenlp")
CORENLP_VERSION = os.environ.get("CORENLP_VERSION", "4.5.10")
INSTALL_CORENLP_IF_MISSING = True
CORENLP_PORT = int(os.environ.get("CORENLP_PORT") or find_open_port())
CORENLP_ENDPOINT = f"http://localhost:{CORENLP_PORT}"
CORENLP_PROPERTIES = "german"

corenlp_path = Path(CORENLP_HOME)

def detect_corenlp_version(path):
    for jar_path in path.glob("stanford-corenlp-*.jar"):
        match = re.fullmatch(r"stanford-corenlp-(\d+\.\d+\.\d+)\.jar", jar_path.name)
        if match:
            return match.group(1)
    return None

detected_version = detect_corenlp_version(corenlp_path)
if detected_version:
    CORENLP_VERSION = detected_version

has_corenlp_jars = detected_version is not None
has_german_models = any(corenlp_path.glob("*german*.jar"))

if not has_corenlp_jars and INSTALL_CORENLP_IF_MISSING:
    corenlp_path.mkdir(parents=True, exist_ok=True)
    stanza.install_corenlp(dir=str(corenlp_path))
    detected_version = detect_corenlp_version(corenlp_path)
    if detected_version:
        CORENLP_VERSION = detected_version
    has_corenlp_jars = detected_version is not None

if not has_german_models and INSTALL_CORENLP_IF_MISSING:
    stanza.download_corenlp_models(model="german", version=CORENLP_VERSION, dir=str(corenlp_path))
    has_german_models = any(corenlp_path.glob("*german*.jar"))

if has_corenlp_jars and has_german_models:
    os.environ["CORENLP_HOME"] = str(corenlp_path)
    CORENLP_HOME = str(corenlp_path)
    print(f"Using CoreNLP {CORENLP_VERSION} from: {display_path(CORENLP_HOME)}")
else:
    CORENLP_HOME = ""
    print("CoreNLP or the German model jar is unavailable. Configure CORENLP_HOME or enable installation in this setup cell.")
    print(f"Main CoreNLP jars found: {has_corenlp_jars}")
    print(f"German model jars found: {has_german_models}")


In [ ]:
# Normalizations observed in the CoreNLP German parse output for these stories.
SYNTAX_EXCEPTIONS = {
    "-LRB-": "(",
    "-RRB-": ")",
    "2-3--1": "2-3",
    "12-13--1": "12-13",
    "13-14--1": "13-14",
}


def count_closed_nodes(tree):
    tree_string = str(tree).replace("\n", "").replace(" ", "")
    runs = []
    current_run = 0

    for char in tree_string:
        if char == ")":
            current_run += 1
        elif current_run:
            runs.append(current_run)
            current_run = 0

    if current_run:
        runs.append(current_run)

    assert tree_string.count(")") == sum(runs)
    return runs


def get_treepositions(tree):
    if isinstance(tree, str):
        tree = Tree.fromstring(tree)

    tree_positions = [list(pos) for pos in tree.treepositions("leaves")]
    labels = [subtree[0] for subtree in tree.subtrees(lambda t: t.height() == 2)]

    assert len(tree_positions) == len(labels)
    return tree_positions, labels


def get_depth(tree_positions):
    return [len(position) for position in tree_positions]


def get_dependencies(sent):
    dependencies = sent["basicDependencies"]
    df_dep = pd.DataFrame(
        {
            "dependency_type": dep["dep"],
            "dependentGloss": dep["dependentGloss"],
            "dependent_idx": dep["dependent"],
            "governor_idx": dep["governor"],
            "governorGloss": dep["governorGloss"],
        }
        for dep in dependencies
    ).sort_values("dependent_idx").reset_index(drop=True)

    surface_words = [token["word"] for token in sent["tokens"]]
    assert list(df_dep["dependentGloss"]) == surface_words

    df_dep["dep_distance"] = df_dep["dependent_idx"] - df_dep["governor_idx"]
    df_dep["absolute_dependency_distance"] = df_dep["dep_distance"].abs()
    return df_dep


def get_phrase_structure_measures(sent):
    tree = Tree.fromstring(sent["parse"])

    closing_nodes = count_closed_nodes(tree)
    tree_positions, word_labels = get_treepositions(tree)
    tree_depth = get_depth(tree_positions)
    word_index = [i for i in range(len(word_labels))]

    df_phrase = pd.DataFrame({
        "word_labels": word_labels,
        "word_index": word_index,
        "closing_nodes": closing_nodes,
        "tree_depth": tree_depth,
    })

    df_phrase["open_nodes"] = df_phrase["tree_depth"] - df_phrase["closing_nodes"]
    return df_phrase


In [ ]:
def analyze_syntax(texts):
    if not CORENLP_HOME:
        return (
            pd.DataFrame([{"status": "Syntax metrics unavailable because CoreNLP is not configured."}]),
            pd.DataFrame(),
            pd.DataFrame(),
        )

    items = normalize_input_texts(texts)
    token_rows = []
    sentence_rows = []

    client = CoreNLPClient(
        annotators=["tokenize", "ssplit", "pos", "lemma", "parse", "depparse"],
        memory="8G",
        endpoint=CORENLP_ENDPOINT,
        timeout=180000,
        be_quiet=True,
        properties=CORENLP_PROPERTIES,
        output_format="json",
        threads=1,
    )

    try:
        for text_id, text in items:
            if len(text) == 0:
                continue

            document = client.annotate(text)

            for sent_index, sent in enumerate(document["sentences"]):
                df_phrase = get_phrase_structure_measures(sent)
                df_phrase["phrase_structure"] = sent["parse"]
                df_phrase = df_phrase.replace(SYNTAX_EXCEPTIONS)

                df_dep = get_dependencies(sent)

                assert list(df_phrase.word_labels) == list(df_dep.dependentGloss)

                df = pd.concat([df_phrase, df_dep], axis=1)
                df["text_id"] = text_id
                df["sent_index"] = sent_index
                df["sentence_length"] = len(sent["tokens"])
                token_rows.append(df)

                sentence_rows.append({
                    "text_id": text_id,
                    "sent_index": sent_index,
                    "sentence_length": len(sent["tokens"]),
                })
    finally:
        client.stop()

    syntax_per_token = pd.concat(token_rows, ignore_index=True) if token_rows else pd.DataFrame()
    syntax_per_sentence = pd.DataFrame(sentence_rows)

    syntax_summary = pd.DataFrame([{
        "Total texts": syntax_per_token["text_id"].nunique() if not syntax_per_token.empty else 0,
        "Total sentences": len(syntax_per_sentence),
        "Median sentence length": safe_median(syntax_per_sentence["sentence_length"].tolist()) if not syntax_per_sentence.empty else 0.0,
        "Median open nodes": safe_median(syntax_per_token["open_nodes"].tolist()) if not syntax_per_token.empty else 0.0,
        "Median tree depth": safe_median(syntax_per_token["tree_depth"].tolist()) if not syntax_per_token.empty else 0.0,
        "Median closed nodes": safe_median(syntax_per_token["closing_nodes"].tolist()) if not syntax_per_token.empty else 0.0,
        "Median absolute dependency distance": safe_median(syntax_per_token["absolute_dependency_distance"].tolist()) if not syntax_per_token.empty else 0.0,
    }]).round(3)

    if syntax_per_token.empty:
        syntax_per_text = pd.DataFrame()
    else:
        syntax_per_text = (
            syntax_per_token.groupby("text_id", as_index=False)
            .agg(
                median_open_nodes=("open_nodes", "median"),
                median_tree_depth=("tree_depth", "median"),
                median_closed_nodes=("closing_nodes", "median"),
                median_absolute_dependency_distance=("absolute_dependency_distance", "median"),
            )
            .merge(
                syntax_per_sentence.groupby("text_id", as_index=False).agg(
                    median_sentence_length=("sentence_length", "median"),
                    sentences=("sent_index", "count"),
                ),
                on="text_id",
            )
            .round(3)
        )

    save_columns = [
        "text_id", "sent_index", "word_labels", "word_index", "closing_nodes",
        "tree_depth", "open_nodes", "phrase_structure", "dependency_type",
        "dependentGloss", "dependent_idx", "governor_idx", "governorGloss",
        "dep_distance", "absolute_dependency_distance", "sentence_length",
    ]
    if not syntax_per_token.empty:
        syntax_per_token = syntax_per_token[save_columns]

    return syntax_summary, syntax_per_text, syntax_per_token


syntax_summary, syntax_per_text, syntax_per_token = analyze_syntax(TEXTS)
display(syntax_summary)
display(syntax_per_text)
display(syntax_per_token.head())


## Semantic similarity

This section implements the two semantic similarity measures. Lexical-level similarity uses Representational Similarity Analysis (RSA): contextual word embeddings are extracted with `google-bert/bert-base-german-cased`, cosine-distance representational dissimilarity matrices are built over shared word types, and Spearman correlations compare the two matrices. Text-level similarity uses sentence embeddings from `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Sentence embeddings are averaged to obtain one normalized vector per text, and the two text vectors are compared with cosine similarity.

In [ ]:
import hashlib
import random
import warnings
from collections import defaultdict

import numpy as np
import torch
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer

LEXICAL_SEMANTIC_MODEL = "google-bert/bert-base-german-cased"
TEXT_SEMANTIC_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

SEMANTIC_SEED = 42
LEXICAL_RSA_ITERATIONS = 10
MIN_SHARED_WORD_TYPES = 3
MAX_BERT_TOKENS = 256

random.seed(SEMANTIC_SEED)
np.random.seed(SEMANTIC_SEED)
torch.manual_seed(SEMANTIC_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Semantic model device:", device)

bert_tokenizer = AutoTokenizer.from_pretrained(LEXICAL_SEMANTIC_MODEL, use_fast=True)
bert_model = AutoModel.from_pretrained(LEXICAL_SEMANTIC_MODEL).to(device)
bert_model.eval()

sbert_model = SentenceTransformer(TEXT_SEMANTIC_MODEL, device=device)


In [ ]:
def require_exactly_two_texts(texts):
    items = normalize_input_texts(texts)
    if len(items) != 2:
        raise ValueError(
            f"Semantic similarity expects exactly two texts in TEXTS, but found {len(items)}."
        )
    return items


def spacy_sentence_wordlists(text, keep_pos=None):
    sent_wordlists = []
    doc = nlp(text)

    for sent in doc.sents:
        words = []
        for token in sent:
            if not token.is_alpha:
                continue
            if keep_pos is not None and token.pos_ not in keep_pos:
                continue
            if token.pos_ in ("NOUN", "PROPN"):
                words.append(token.text)
            else:
                words.append(token.text.lower())
        if words:
            sent_wordlists.append(words)

    return sent_wordlists


def mean_last_four_layers(hidden_states):
    stacked_layers = torch.stack(hidden_states[-4:], dim=0)
    return stacked_layers.mean(dim=0)[0].detach().cpu().numpy().astype(np.float32)


def collect_word_embeddings(text, max_tokens=256):
    embeddings_by_word = defaultdict(list)

    for words in spacy_sentence_wordlists(text):
        encoded = bert_tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=max_tokens,
            padding=False,
            return_tensors="pt",
        )
        model_inputs = {key: value.to(device) for key, value in encoded.items()}

        with torch.inference_mode():
            output = bert_model(**model_inputs, output_hidden_states=True)

        token_vecs = mean_last_four_layers(output.hidden_states)
        word_ids = encoded.word_ids(batch_index=0)
        token_positions_by_word = defaultdict(list)

        for token_i, word_i in enumerate(word_ids):
            if word_i is not None:
                token_positions_by_word[word_i].append(token_i)

        for word_i, token_positions in token_positions_by_word.items():
            word = words[word_i]
            vec = token_vecs[token_positions].mean(axis=0).astype(np.float32)
            embeddings_by_word[word].append(vec)

    return embeddings_by_word


def cosine_similarity_matrix(vectors):
    matrix = vectors.astype(np.float32)
    matrix = matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-8)
    return matrix @ matrix.T


def upper_triangle_flat(matrix):
    indices = np.triu_indices(matrix.shape[0], k=1)
    return matrix[indices]


def stable_seed(*parts):
    seed_material = "|".join(str(part) for part in parts)
    return int(hashlib.md5(seed_material.encode("utf-8")).hexdigest()[:8], 16)


def lexical_rsa_similarity(word_embs_a, word_embs_b, iterations=10, min_shared_words=3):
    shared_words = sorted(set(word_embs_a).intersection(word_embs_b))
    n_shared = len(shared_words)

    if n_shared < min_shared_words:
        return np.nan, np.nan, 0, n_shared, shared_words

    rng = random.Random(stable_seed(SEMANTIC_SEED, "lexical_rsa", *shared_words))
    correlations = []

    for _sample in range(iterations):
        vecs_a = []
        vecs_b = []

        for word in shared_words:
            vecs_a.append(rng.choice(word_embs_a[word]))
            vecs_b.append(rng.choice(word_embs_b[word]))

        if len(vecs_a) < min_shared_words:
            continue

        sim_a = cosine_similarity_matrix(np.vstack(vecs_a))
        sim_b = cosine_similarity_matrix(np.vstack(vecs_b))

        flat_a = upper_triangle_flat(sim_a)
        flat_b = upper_triangle_flat(sim_b)

        if len(flat_a) < 2:
            continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r, _ = spearmanr(flat_a, flat_b)

        if not np.isnan(r):
            correlations.append(float(r))

    if not correlations:
        return np.nan, np.nan, 0, n_shared, shared_words

    return (
        float(np.mean(correlations)),
        float(np.std(correlations, ddof=1)) if len(correlations) > 1 else 0.0,
        len(correlations),
        n_shared,
        shared_words,
    )


def text_embedding_from_spacy_sentences(text, batch_size=32):
    sent_wordlists = spacy_sentence_wordlists(text)
    sent_strings = [" ".join(words) for words in sent_wordlists]

    if not sent_strings:
        return None

    sent_embs = sbert_model.encode(
        sent_strings,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    text_emb = sent_embs.mean(axis=0)
    text_emb = text_emb / (np.linalg.norm(text_emb) + 1e-8)
    return text_emb.astype(np.float32)


def cosine_between_normalized(vec_a, vec_b):
    if vec_a is None or vec_b is None:
        return np.nan
    return float(np.dot(vec_a, vec_b))


In [ ]:
def analyze_semantic_similarity(texts):
    (label_a, text_a), (label_b, text_b) = require_exactly_two_texts(texts)

    word_embs_a = collect_word_embeddings(text_a, max_tokens=MAX_BERT_TOKENS)
    word_embs_b = collect_word_embeddings(text_b, max_tokens=MAX_BERT_TOKENS)

    lexical_mean, lexical_sd, valid_iterations, n_shared, shared_words = lexical_rsa_similarity(
        word_embs_a,
        word_embs_b,
        iterations=LEXICAL_RSA_ITERATIONS,
        min_shared_words=MIN_SHARED_WORD_TYPES,
    )

    text_emb_a = text_embedding_from_spacy_sentences(text_a)
    text_emb_b = text_embedding_from_spacy_sentences(text_b)
    text_level_similarity = cosine_between_normalized(text_emb_a, text_emb_b)

    semantic_summary = pd.DataFrame([
        {
            "measure": "lexical_level_semantic_similarity_rsa",
            "text_a": label_a,
            "text_b": label_b,
            "similarity": lexical_mean,
            "sd_across_iterations": lexical_sd,
            "valid_iterations": valid_iterations,
            "shared_word_types": n_shared,
            "minimum_shared_word_types": MIN_SHARED_WORD_TYPES,
            "model": LEXICAL_SEMANTIC_MODEL,
        },
        {
            "measure": "text_level_semantic_similarity_cosine",
            "text_a": label_a,
            "text_b": label_b,
            "similarity": text_level_similarity,
            "sd_across_iterations": np.nan,
            "valid_iterations": np.nan,
            "shared_word_types": np.nan,
            "minimum_shared_word_types": np.nan,
            "model": TEXT_SEMANTIC_MODEL,
        },
    ]).round(4)

    shared_words_df = pd.DataFrame({"shared_word_type": shared_words})
    return semantic_summary, shared_words_df


semantic_summary, semantic_shared_words = analyze_semantic_similarity(TEXTS)

display(semantic_summary)
display(semantic_shared_words)
